In [1]:
# %%
from pathlib import Path
import pickle
import json
import joblib

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)

from xgboost import XGBClassifier


# =========================
# 1. 路径配置
# =========================
ROOT = Path("/home/lixiangyu/xwq/Subcellular_Feature")

PKL_PATH = ROOT / "data" / "simulated_data_dict.pkl"
FEAT_PATH = ROOT / "data" / "simulated_features.parquet"

OUT_DIR = ROOT / "data" / "xgb_7class_no_foci"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OOF_OUT_PARQ = OUT_DIR / "multiclass_xgb_7class_no_foci_group5cv_oof.parquet"
FOLD_METRICS_CSV = OUT_DIR / "multiclass_xgb_7class_no_foci_group5cv_fold_metrics.csv"
SUMMARY_CSV = OUT_DIR / "multiclass_xgb_7class_no_foci_group5cv_summary.csv"
PER_CLASS_CSV = OUT_DIR / "multiclass_xgb_7class_no_foci_group5cv_per_class_metrics.csv"
OVERALL_JSON = OUT_DIR / "multiclass_xgb_7class_no_foci_group5cv_overall_metrics.json"

CM_DIR = OUT_DIR / "confusion_matrices"
CM_DIR.mkdir(parents=True, exist_ok=True)

FINAL_OUT_PARQ = OUT_DIR / "multiclass_xgb_7class_no_foci_final_predictions.parquet"
FINAL_MODEL_PATH = ROOT / "models" / "multiclass_xgb_7class_no_foci_final_from_cv.joblib"

print("ROOT:", ROOT)
print("PKL_PATH:", PKL_PATH)
print("FEAT_PATH:", FEAT_PATH)
print("OUT_DIR:", OUT_DIR)
print("FINAL_MODEL_PATH:", FINAL_MODEL_PATH)

ROOT: /home/lixiangyu/xwq/Subcellular_Feature
PKL_PATH: /home/lixiangyu/xwq/Subcellular_Feature/data/simulated_data_dict.pkl
FEAT_PATH: /home/lixiangyu/xwq/Subcellular_Feature/data/simulated_features.parquet
OUT_DIR: /home/lixiangyu/xwq/Subcellular_Feature/data/xgb_7class_no_foci
FINAL_MODEL_PATH: /home/lixiangyu/xwq/Subcellular_Feature/models/multiclass_xgb_7class_no_foci_final_from_cv.joblib


In [2]:
# %%
# =========================
# 2. 读取模拟数据与特征，构造 no-Foci 标签
# =========================
with open(PKL_PATH, "rb") as f:
    obj = pickle.load(f)

spot = obj["data_df"].copy()
feat = pd.read_parquet(FEAT_PATH).copy()

# 统一 cell/gene/groundtruth 类型
spot["cell"] = spot["cell"].astype(str).str.strip()
spot["gene"] = spot["gene"].astype(str).str.strip()
spot["groundtruth"] = spot["groundtruth"].astype(str).str.strip()

feat["cell"] = feat["cell"].astype(str).str.strip()
feat["gene"] = feat["gene"].astype(str).str.strip()

print("raw spot shape:", spot.shape)
print("raw feature shape:", feat.shape)
print("raw transcript-level groundtruth counts:")
print(spot["groundtruth"].value_counts())

# 删除所有 Foci transcript
spot_no_foci = spot[spot["groundtruth"] != "Foci"].copy()

print("\nafter removing Foci transcript shape:", spot_no_foci.shape)
print("transcript-level groundtruth counts after removing Foci:")
print(spot_no_foci["groundtruth"].value_counts())

# transcript -> cell-gene label
y_df = (
    spot_no_foci
    .groupby(["cell", "gene"], as_index=False)["groundtruth"]
    .first()
    .rename(columns={"groundtruth": "y_true"})
)

# 合并 17 维特征
data = feat.merge(y_df, on=["cell", "gene"], how="inner").copy()

print("\ny_df shape:", y_df.shape)
print("merged data shape:", data.shape)
print("cell-gene label counts:")
print(data["y_true"].value_counts())

assert "Foci" not in set(data["y_true"].astype(str)), "Foci still exists in training data!"
data.head()

raw spot shape: (1347881, 12)
raw feature shape: (20000, 19)
raw transcript-level groundtruth counts:
groundtruth
Random          187500
Nuclear         187500
Nuclear edge    187500
Cell edge       187500
Protrusion      187500
Radial          187500
Cytoplasmic     118379
Foci            104502
Name: count, dtype: int64

after removing Foci transcript shape: (1243379, 12)
transcript-level groundtruth counts after removing Foci:
groundtruth
Random          187500
Nuclear         187500
Nuclear edge    187500
Cell edge       187500
Protrusion      187500
Radial          187500
Cytoplasmic     118379
Name: count, dtype: int64

y_df shape: (17500, 3)
merged data shape: (17500, 20)
cell-gene label counts:
y_true
Random          2500
Nuclear         2500
Cytoplasmic     2500
Nuclear edge    2500
Cell edge       2500
Protrusion      2500
Radial          2500
Name: count, dtype: int64


,cell,gene,bento_cell_inner_proximity,bento_nucleus_inner_proximity,bento_nucleus_outer_proximity,bento_cell_inner_asymmetry,bento_nucleus_inner_asymmetry,bento_nucleus_outer_asymmetry,bento_point_dispersion_norm,bento_nucleus_dispersion_norm,bento_l_max,bento_l_max_gradient,bento_l_min_gradient,bento_l_monotony,bento_l_half_radius,sprawl_peripheral,sprawl_central,sprawl_punctate,sprawl_radial,y_true
0,cell_102,gene_0_0_0,0.650084,0.786023,0.754027,0.562135,0.316887,0.789136,0.664003,0.799773,3.076283,1.012939,-0.880864,-0.937887,-2.462147,-0.173893,0.163147,0.07,-0.45,Random
1,cell_102,gene_0_0_1,0.654681,0.823567,0.698191,0.617036,0.362271,0.853534,0.868480,0.832679,3.255770,0.579378,-0.708565,-0.942784,-5.598550,-0.150597,0.183738,0.32,0.46,Random
2,cell_102,gene_0_0_2,0.661598,0.820110,0.736471,0.571356,0.356906,0.803288,0.648734,0.634645,3.345068,0.428725,-0.758260,-0.934486,-2.227309,-0.158413,0.177275,-0.01,-0.40,Random
3,cell_102,gene_0_0_3,0.667682,0.832543,0.755766,0.589740,0.368915,0.789935,0.739277,0.689100,0.964089,0.240149,-0.761459,-0.939861,-5.406958,-0.088825,0.175810,0.03,-1.00,Random
4,cell_102,gene_0_0_4,0.657550,0.804136,0.713734,0.591729,0.345639,0.833883,0.765556,0.719655,0.736492,0.289656,-0.724002,-0.960981,-5.511314,-0.140678,0.146013,0.10,-0.25,Random


In [3]:
# %%
# =========================
# 3. 类别与特征列
# =========================
CLASSES = [
    "Nuclear",
    "Cytoplasmic",
    "Nuclear edge",
    "Cell edge",
    "Protrusion",
    "Radial",
    "Random",
]

# 只保留 7 类，防止标签源里有其它异常类别
data = data[data["y_true"].isin(CLASSES)].copy()

FEATURE_COLS = [
    c for c in data.columns
    if c.startswith("bento_") or c.startswith("sprawl_")
]

CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}
ID_TO_CLASS = {i: c for c, i in CLASS_TO_ID.items()}

print("CLASSES:", CLASSES)
print("n_classes:", len(CLASSES))
print("class_to_id:", CLASS_TO_ID)
print("n_features:", len(FEATURE_COLS))
print(FEATURE_COLS)
print("\nfinal training class distribution:")
print(data["y_true"].value_counts())

assert len(CLASSES) == 7
assert "Foci" not in CLASSES
assert "Foci" not in set(data["y_true"].astype(str))

CLASSES: ['Nuclear', 'Cytoplasmic', 'Nuclear edge', 'Cell edge', 'Protrusion', 'Radial', 'Random']
n_classes: 7
class_to_id: {'Nuclear': 0, 'Cytoplasmic': 1, 'Nuclear edge': 2, 'Cell edge': 3, 'Protrusion': 4, 'Radial': 5, 'Random': 6}
n_features: 17
['bento_cell_inner_proximity', 'bento_nucleus_inner_proximity', 'bento_nucleus_outer_proximity', 'bento_cell_inner_asymmetry', 'bento_nucleus_inner_asymmetry', 'bento_nucleus_outer_asymmetry', 'bento_point_dispersion_norm', 'bento_nucleus_dispersion_norm', 'bento_l_max', 'bento_l_max_gradient', 'bento_l_min_gradient', 'bento_l_monotony', 'bento_l_half_radius', 'sprawl_peripheral', 'sprawl_central', 'sprawl_punctate', 'sprawl_radial']

final training class distribution:
y_true
Random          2500
Nuclear         2500
Cytoplasmic     2500
Nuclear edge    2500
Cell edge       2500
Protrusion      2500
Radial          2500
Name: count, dtype: int64


In [4]:
# %%
# =========================
# 4. XGBoost 模型构造函数
# =========================
def make_multiclass_xgb(random_state=42):
    model = XGBClassifier(
        objective="multi:softprob",
        num_class=len(CLASSES),
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        min_child_weight=1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        eval_metric="mlogloss",
        random_state=random_state,
        n_jobs=-1,
        tree_method="hist",
    )
    return model

In [5]:
# %%
# =========================
# 5. 训练输入与分组交叉验证
# =========================
X = data[FEATURE_COLS].copy()
y = data["y_true"].astype(str).copy()
y_id = y.map(CLASS_TO_ID).astype(int)
groups = data["cell"].astype(str).values

gkf = GroupKFold(n_splits=5)

fold_metrics = []
oof_rows = []
fold_confusion_matrices = {}

print("X:", X.shape)
print("y:", y_id.shape)
print("n_cells:", data["cell"].nunique())
print("n_genes:", data["gene"].nunique())

X: (17500, 17)
y: (17500,)
n_cells: 50
n_genes: 350


In [6]:
# %%
# =========================
# 6. 5-fold GroupKFold OOF evaluation
# =========================
for fold_id, (train_idx, test_idx) in enumerate(gkf.split(X, y_id, groups=groups), start=1):
    print(f"\n========== Fold {fold_id} ==========")

    X_train = X.iloc[train_idx].reset_index(drop=True)
    y_train = y_id.iloc[train_idx].reset_index(drop=True)
    cell_train = data["cell"].iloc[train_idx].reset_index(drop=True)

    X_test = X.iloc[test_idx].reset_index(drop=True)
    y_test = y_id.iloc[test_idx].reset_index(drop=True)
    cell_test = data["cell"].iloc[test_idx].reset_index(drop=True)

    print("train:", X_train.shape, "test:", X_test.shape)
    print("train cells:", cell_train.nunique(), "test cells:", cell_test.nunique())

    model = make_multiclass_xgb(random_state=42 + fold_id)
    model.fit(X_train, y_train)

    prob_test = model.predict_proba(X_test)
    pred_test_id = prob_test.argmax(axis=1)
    pred_test = pd.Series(pred_test_id).map(ID_TO_CLASS).values
    y_test_label = pd.Series(y_test).map(ID_TO_CLASS).values

    prob_df = pd.DataFrame(
        prob_test,
        columns=[f"p_{ID_TO_CLASS[i]}" for i in range(len(CLASSES))]
    )
    prob_df = prob_df.reindex(columns=[f"p_{c}" for c in CLASSES])

    fold_out = data.iloc[test_idx][["cell", "gene", "y_true"] + FEATURE_COLS].reset_index(drop=True).copy()

    for c in CLASSES:
        fold_out[f"p_{c}"] = prob_df[f"p_{c}"].values

    fold_out["pattern_top1"] = pred_test
    fold_out["fold"] = fold_id
    oof_rows.append(fold_out)

    acc = accuracy_score(y_test_label, pred_test)
    bal_acc = balanced_accuracy_score(y_test_label, pred_test)

    report = classification_report(
        y_test_label,
        pred_test,
        labels=CLASSES,
        output_dict=True,
        zero_division=0,
    )

    row = {
        "fold": fold_id,
        "n_train": len(train_idx),
        "n_test": len(test_idx),
        "n_train_cells": cell_train.nunique(),
        "n_test_cells": cell_test.nunique(),
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "macro_precision": report["macro avg"]["precision"],
        "macro_recall": report["macro avg"]["recall"],
        "macro_f1": report["macro avg"]["f1-score"],
        "weighted_precision": report["weighted avg"]["precision"],
        "weighted_recall": report["weighted avg"]["recall"],
        "weighted_f1": report["weighted avg"]["f1-score"],
    }

    for c in CLASSES:
        row[f"precision_{c}"] = report[c]["precision"]
        row[f"recall_{c}"] = report[c]["recall"]
        row[f"f1_{c}"] = report[c]["f1-score"]
        row[f"support_{c}"] = report[c]["support"]

    fold_metrics.append(row)

    cm = confusion_matrix(y_test_label, pred_test, labels=CLASSES)
    cm_df = pd.DataFrame(
        cm,
        index=[f"true:{c}" for c in CLASSES],
        columns=[f"pred:{c}" for c in CLASSES],
    )
    fold_confusion_matrices[fold_id] = cm_df

    cm_path = CM_DIR / f"fold{fold_id}_confusion_matrix.csv"
    cm_df.to_csv(cm_path, encoding="utf-8-sig")

    print(f"fold {fold_id} accuracy:", acc)
    print(f"fold {fold_id} balanced_accuracy:", bal_acc)
    print(f"saved confusion matrix:", cm_path)


========== Fold 1 ==========
train: (14000, 17) test: (3500, 17)
train cells: 40 test cells: 10
fold 1 accuracy: 0.922
fold 1 balanced_accuracy: 0.922
saved confusion matrix: /home/lixiangyu/xwq/Subcellular_Feature/data/xgb_7class_no_foci/confusion_matrices/fold1_confusion_matrix.csv

========== Fold 2 ==========
train: (14000, 17) test: (3500, 17)
train cells: 40 test cells: 10
fold 2 accuracy: 0.9214285714285714
fold 2 balanced_accuracy: 0.9214285714285715
saved confusion matrix: /home/lixiangyu/xwq/Subcellular_Feature/data/xgb_7class_no_foci/confusion_matrices/fold2_confusion_matrix.csv

========== Fold 3 ==========
train: (14000, 17) test: (3500, 17)
train cells: 40 test cells: 10
fold 3 accuracy: 0.938
fold 3 balanced_accuracy: 0.938
saved confusion matrix: /home/lixiangyu/xwq/Subcellular_Feature/data/xgb_7class_no_foci/confusion_matrices/fold3_confusion_matrix.csv

========== Fold 4 ==========
train: (14000, 17) test: (3500, 17)
train cells: 40 test cells: 10
fold 4 accuracy: 0.

In [7]:
# %%
# =========================
# 7. 保存 OOF 与 fold metrics
# =========================
fold_metrics_df = pd.DataFrame(fold_metrics)
oof_df = pd.concat(oof_rows, axis=0, ignore_index=True)

oof_df = oof_df.sort_values(["fold", "cell", "gene"]).reset_index(drop=True)

fold_metrics_df.to_csv(FOLD_METRICS_CSV, index=False, encoding="utf-8-sig")
oof_df.to_parquet(OOF_OUT_PARQ, index=False)

print("saved:", FOLD_METRICS_CSV)
print("saved:", OOF_OUT_PARQ)
print("oof_df shape:", oof_df.shape)
oof_df.head()

saved: /home/lixiangyu/xwq/Subcellular_Feature/data/xgb_7class_no_foci/multiclass_xgb_7class_no_foci_group5cv_fold_metrics.csv
saved: /home/lixiangyu/xwq/Subcellular_Feature/data/xgb_7class_no_foci/multiclass_xgb_7class_no_foci_group5cv_oof.parquet
oof_df shape: (17500, 29)


,cell,gene,y_true,bento_cell_inner_proximity,bento_nucleus_inner_proximity,bento_nucleus_outer_proximity,bento_cell_inner_asymmetry,bento_nucleus_inner_asymmetry,bento_nucleus_outer_asymmetry,bento_point_dispersion_norm,...,sprawl_radial,p_Nuclear,p_Cytoplasmic,p_Nuclear edge,p_Cell edge,p_Protrusion,p_Radial,p_Random,pattern_top1,fold
0,cell_135,gene_0_0_0,Random,0.676920,0.789751,0.743663,0.490438,0.304394,0.786706,0.584157,...,-0.13,0.006336,0.161243,0.178318,0.000673,0.000667,0.006519,0.646245,Random,1
1,cell_135,gene_0_0_1,Random,0.708254,0.837672,0.695342,0.635294,0.345390,0.865626,0.833569,...,0.67,0.000221,0.654452,0.001414,0.000346,0.000198,0.000258,0.343109,Cytoplasmic,1
2,cell_135,gene_0_0_2,Random,0.694508,0.840011,0.741575,0.584942,0.343822,0.799803,0.709921,...,0.70,0.000068,0.086051,0.000688,0.000134,0.000071,0.000083,0.912906,Random,1
3,cell_135,gene_0_0_3,Random,0.703409,0.824135,0.717576,0.594135,0.299609,0.834262,0.723111,...,-0.37,0.000067,0.058808,0.000251,0.000056,0.000058,0.000067,0.940693,Random,1
4,cell_135,gene_0_0_4,Random,0.693703,0.819843,0.703387,0.567039,0.318853,0.841895,0.692065,...,0.71,0.000087,0.034611,0.001192,0.000070,0.000072,0.000087,0.963880,Random,1


In [8]:
# %%
# =========================
# 8. OOF overall metrics
# =========================
oof_acc = (oof_df["y_true"] == oof_df["pattern_top1"]).mean()
oof_bal_acc = balanced_accuracy_score(oof_df["y_true"], oof_df["pattern_top1"])

oof_report = classification_report(
    oof_df["y_true"],
    oof_df["pattern_top1"],
    labels=CLASSES,
    output_dict=True,
    zero_division=0,
)

overall_metrics = {
    "oof_accuracy": float(oof_acc),
    "oof_balanced_accuracy": float(oof_bal_acc),
    "macro_precision": float(oof_report["macro avg"]["precision"]),
    "macro_recall": float(oof_report["macro avg"]["recall"]),
    "macro_f1": float(oof_report["macro avg"]["f1-score"]),
    "weighted_precision": float(oof_report["weighted avg"]["precision"]),
    "weighted_recall": float(oof_report["weighted avg"]["recall"]),
    "weighted_f1": float(oof_report["weighted avg"]["f1-score"]),
}

with open(OVERALL_JSON, "w", encoding="utf-8") as f:
    json.dump(overall_metrics, f, ensure_ascii=False, indent=2)

print("saved:", OVERALL_JSON)
print(json.dumps(overall_metrics, indent=2))

saved: /home/lixiangyu/xwq/Subcellular_Feature/data/xgb_7class_no_foci/multiclass_xgb_7class_no_foci_group5cv_overall_metrics.json
{
  "oof_accuracy": 0.9274857142857142,
  "oof_balanced_accuracy": 0.9274857142857142,
  "macro_precision": 0.9281157339208163,
  "macro_recall": 0.9274857142857142,
  "macro_f1": 0.927716262379012,
  "weighted_precision": 0.9281157339208163,
  "weighted_recall": 0.9274857142857142,
  "weighted_f1": 0.9277162623790121
}


In [9]:
# %%
# =========================
# 9. fold summary
# =========================
metric_cols = [
    "accuracy",
    "balanced_accuracy",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "weighted_precision",
    "weighted_recall",
    "weighted_f1",
] + [f"f1_{c}" for c in CLASSES]

summary_df = pd.DataFrame({
    "metric": metric_cols,
    "mean": [fold_metrics_df[c].mean() for c in metric_cols],
    "std": [fold_metrics_df[c].std(ddof=1) for c in metric_cols],
})

summary_df.to_csv(SUMMARY_CSV, index=False, encoding="utf-8-sig")
print("saved:", SUMMARY_CSV)
summary_df

saved: /home/lixiangyu/xwq/Subcellular_Feature/data/xgb_7class_no_foci/multiclass_xgb_7class_no_foci_group5cv_summary.csv


,metric,mean,std
0,accuracy,0.927486,0.008615
1,balanced_accuracy,0.927486,0.008615
2,macro_precision,0.929139,0.007363
3,macro_recall,0.927486,0.008615
4,macro_f1,0.927772,0.008304
5,weighted_precision,0.929139,0.007363
6,weighted_recall,0.927486,0.008615
7,weighted_f1,0.927772,0.008304
8,f1_Nuclear,0.987186,0.004454
9,f1_Cytoplasmic,0.843636,0.019431


In [11]:
# %%
# =========================
# 10. per-class metrics
# =========================
per_class_rows = []
for c in CLASSES:
    per_class_rows.append({
        "class": c,
        "precision": oof_report[c]["precision"],
        "recall": oof_report[c]["recall"],
        "f1": oof_report[c]["f1-score"],
        "support": oof_report[c]["support"],
    })

per_class_df = pd.DataFrame(per_class_rows)
per_class_df.to_csv(PER_CLASS_CSV, index=False, encoding="utf-8-sig")
print("saved:", PER_CLASS_CSV)
per_class_df

saved: /home/lixiangyu/xwq/Subcellular_Feature/data/xgb_7class_no_foci/multiclass_xgb_7class_no_foci_group5cv_per_class_metrics.csv


,class,precision,recall,f1,support
0,Nuclear,0.987981,0.9864,0.987190,2500.0
1,Cytoplasmic,0.847677,0.8392,0.843417,2500.0
2,Nuclear edge,0.931232,0.9100,0.920494,2500.0
3,Cell edge,0.933829,0.9540,0.943807,2500.0
4,Protrusion,0.994715,0.9788,0.986694,2500.0
5,Radial,0.977438,0.9704,0.973906,2500.0
6,Random,0.823938,0.8536,0.838507,2500.0


In [12]:
# %%
# =========================
# 11. OOF confusion matrix
# =========================
cm_oof = confusion_matrix(
    oof_df["y_true"],
    oof_df["pattern_top1"],
    labels=CLASSES,
)

cm_oof_df = pd.DataFrame(
    cm_oof,
    index=[f"true:{c}" for c in CLASSES],
    columns=[f"pred:{c}" for c in CLASSES],
)

OOF_CM_CSV = CM_DIR / "oof_confusion_matrix.csv"
cm_oof_df.to_csv(OOF_CM_CSV, encoding="utf-8-sig")
print("saved:", OOF_CM_CSV)
cm_oof_df

saved: /home/lixiangyu/xwq/Subcellular_Feature/data/xgb_7class_no_foci/confusion_matrices/oof_confusion_matrix.csv


,pred:Nuclear,pred:Cytoplasmic,pred:Nuclear edge,pred:Cell edge,pred:Protrusion,pred:Radial,pred:Random
true:Nuclear,2466,0,3,0,0,4,27
true:Cytoplasmic,5,2098,30,90,0,1,276
true:Nuclear edge,6,25,2275,1,0,42,151
true:Cell edge,0,106,0,2385,1,8,0
true:Protrusion,0,0,0,53,2447,0,0
true:Radial,0,7,28,25,12,2426,2
true:Random,19,239,107,0,0,1,2134


In [13]:
# %%
# =========================
# 12. train final model on all no-Foci data
# =========================
X_all = data[FEATURE_COLS].reset_index(drop=True).copy()
y_all = data["y_true"].astype(str).reset_index(drop=True).copy()
y_all_id = y_all.map(CLASS_TO_ID).astype(int)

final_model = make_multiclass_xgb(random_state=42)
final_model.fit(X_all, y_all_id)

print("trained final 7-class no-Foci multiclass XGBoost model")

trained final 7-class no-Foci multiclass XGBoost model


In [14]:
# %%
# =========================
# 13. final model predictions on training table
# =========================
prob_all = final_model.predict_proba(X_all)
pred_all_id = prob_all.argmax(axis=1)
pred_all = pd.Series(pred_all_id).map(ID_TO_CLASS).values

prob_df = pd.DataFrame(
    prob_all,
    columns=[f"p_{ID_TO_CLASS[i]}" for i in range(len(CLASSES))]
)
prob_df = prob_df.reindex(columns=[f"p_{c}" for c in CLASSES])

out = data.copy().reset_index(drop=True)

for c in CLASSES:
    out[f"p_{c}"] = prob_df[f"p_{c}"].values

out["pattern_top1"] = pred_all

base_cols = ["cell", "gene", "y_true"]
prob_cols = [f"p_{c}" for c in CLASSES]
final_cols = base_cols + FEATURE_COLS + prob_cols + ["pattern_top1"]
out = out[final_cols]

FINAL_OUT_PARQ.parent.mkdir(parents=True, exist_ok=True)
out.to_parquet(FINAL_OUT_PARQ, index=False)

print("saved:", FINAL_OUT_PARQ)
print(out.shape)
out.head()

saved: /home/lixiangyu/xwq/Subcellular_Feature/data/xgb_7class_no_foci/multiclass_xgb_7class_no_foci_final_predictions.parquet
(17500, 28)


,cell,gene,y_true,bento_cell_inner_proximity,bento_nucleus_inner_proximity,bento_nucleus_outer_proximity,bento_cell_inner_asymmetry,bento_nucleus_inner_asymmetry,bento_nucleus_outer_asymmetry,bento_point_dispersion_norm,...,sprawl_punctate,sprawl_radial,p_Nuclear,p_Cytoplasmic,p_Nuclear edge,p_Cell edge,p_Protrusion,p_Radial,p_Random,pattern_top1
0,cell_102,gene_0_0_0,Random,0.650084,0.786023,0.754027,0.562135,0.316887,0.789136,0.664003,...,0.07,-0.45,0.000173,0.104190,0.007397,0.000159,0.000151,0.000298,0.887632,Random
1,cell_102,gene_0_0_1,Random,0.654681,0.823567,0.698191,0.617036,0.362271,0.853534,0.868480,...,0.32,0.46,0.000173,0.184518,0.002932,0.000161,0.000147,0.000172,0.811897,Random
2,cell_102,gene_0_0_2,Random,0.661598,0.820110,0.736471,0.571356,0.356906,0.803288,0.648734,...,-0.01,-0.40,0.000253,0.009343,0.001304,0.000052,0.000050,0.000077,0.988922,Random
3,cell_102,gene_0_0_3,Random,0.667682,0.832543,0.755766,0.589740,0.368915,0.789935,0.739277,...,0.03,-1.00,0.000049,0.019095,0.000733,0.000047,0.000041,0.000063,0.979974,Random
4,cell_102,gene_0_0_4,Random,0.657550,0.804136,0.713734,0.591729,0.345639,0.833883,0.765556,...,0.10,-0.25,0.000069,0.030257,0.000343,0.000087,0.000063,0.000071,0.969110,Random


In [15]:
# %%
# =========================
# 14. save final model bundle
# =========================
save_obj = {
    "model": final_model,
    "feature_cols": FEATURE_COLS,
    "classes": CLASSES,
    "class_to_id": CLASS_TO_ID,
    "id_to_class": ID_TO_CLASS,
    "meta": {
        "trained_on": "all available labeled data after removing Foci and after GroupKFold 5-fold CV evaluation",
        "task_type": "single-label multiclass classification",
        "model_type": "xgboost_multiclass",
        "removed_class": "Foci",
        "n_classes": len(CLASSES),
        "n_samples": int(len(data)),
        "n_cells": int(data["cell"].nunique()),
        "n_genes": int(data["gene"].nunique()),
        "model_version": "multiclass_xgb_7class_no_foci_v1",
        "xgb_params": {
            "objective": "multi:softprob",
            "num_class": len(CLASSES),
            "n_estimators": 300,
            "learning_rate": 0.05,
            "max_depth": 6,
            "min_child_weight": 1,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_alpha": 0.0,
            "reg_lambda": 1.0,
            "eval_metric": "mlogloss",
            "random_state": 42,
            "n_jobs": -1,
            "tree_method": "hist",
        },
        "oof_metrics": overall_metrics,
        "fold_metrics": fold_metrics,
    },
}

FINAL_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(save_obj, FINAL_MODEL_PATH)

print("saved final model:", FINAL_MODEL_PATH)

saved final model: /home/lixiangyu/xwq/Subcellular_Feature/models/multiclass_xgb_7class_no_foci_final_from_cv.joblib


In [16]:
# %%
# =========================
# 15. check saved model bundle
# =========================
loaded = joblib.load(FINAL_MODEL_PATH)
print("keys:", loaded.keys())
print("classes:", loaded["classes"])
print("meta:", loaded["meta"])
print("model type:", type(loaded["model"]))

assert "Foci" not in loaded["classes"]
assert loaded["meta"]["n_classes"] == 7
assert loaded["meta"]["removed_class"] == "Foci"

keys: dict_keys(['model', 'feature_cols', 'classes', 'class_to_id', 'id_to_class', 'meta'])
classes: ['Nuclear', 'Cytoplasmic', 'Nuclear edge', 'Cell edge', 'Protrusion', 'Radial', 'Random']
meta: {'trained_on': 'all available labeled data after removing Foci and after GroupKFold 5-fold CV evaluation', 'task_type': 'single-label multiclass classification', 'model_type': 'xgboost_multiclass', 'removed_class': 'Foci', 'n_classes': 7, 'n_samples': 17500, 'n_cells': 50, 'n_genes': 350, 'model_version': 'multiclass_xgb_7class_no_foci_v1', 'xgb_params': {'objective': 'multi:softprob', 'num_class': 7, 'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 6, 'min_child_weight': 1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'eval_metric': 'mlogloss', 'random_state': 42, 'n_jobs': -1, 'tree_method': 'hist'}, 'oof_metrics': {'oof_accuracy': 0.9274857142857142, 'oof_balanced_accuracy': 0.9274857142857142, 'macro_precision': 0.9281157339208163, 'macro_recall'

In [17]:
# %%
# =========================
# 16. quick compatibility check with PatternClassifier
# =========================
from tools.models.pattern_classifier import PatternClassifier

clf = PatternClassifier(FINAL_MODEL_PATH)
print("classes:", clf.classes)
print("n_features:", len(clf.feature_cols))
print("model type:", type(clf.model))

test_pred = clf.predict(data[["cell", "gene"] + FEATURE_COLS].head(5))
test_pred

classes: ['Nuclear', 'Cytoplasmic', 'Nuclear edge', 'Cell edge', 'Protrusion', 'Radial', 'Random']
n_features: 17
model type: <class 'xgboost.sklearn.XGBClassifier'>


,cell,gene,bento_cell_inner_proximity,bento_nucleus_inner_proximity,bento_nucleus_outer_proximity,bento_cell_inner_asymmetry,bento_nucleus_inner_asymmetry,bento_nucleus_outer_asymmetry,bento_point_dispersion_norm,bento_nucleus_dispersion_norm,...,sprawl_punctate,sprawl_radial,p_Nuclear,p_Cytoplasmic,p_Nuclear edge,p_Cell edge,p_Protrusion,p_Radial,p_Random,pattern
0,cell_102,gene_0_0_0,0.650084,0.786023,0.754027,0.562135,0.316887,0.789136,0.664003,0.799773,...,0.07,-0.45,0.000173,0.104190,0.007397,0.000159,0.000151,0.000298,0.887632,Random
1,cell_102,gene_0_0_1,0.654681,0.823567,0.698191,0.617036,0.362271,0.853534,0.868480,0.832679,...,0.32,0.46,0.000173,0.184518,0.002932,0.000161,0.000147,0.000172,0.811897,Random
2,cell_102,gene_0_0_2,0.661598,0.820110,0.736471,0.571356,0.356906,0.803288,0.648734,0.634645,...,-0.01,-0.40,0.000253,0.009343,0.001304,0.000052,0.000050,0.000077,0.988922,Random
3,cell_102,gene_0_0_3,0.667682,0.832543,0.755766,0.589740,0.368915,0.789935,0.739277,0.689100,...,0.03,-1.00,0.000049,0.019095,0.000733,0.000047,0.000041,0.000063,0.979974,Random
4,cell_102,gene_0_0_4,0.657550,0.804136,0.713734,0.591729,0.345639,0.833883,0.765556,0.719655,...,0.10,-0.25,0.000069,0.030257,0.000343,0.000087,0.000063,0.000071,0.969110,Random


In [18]:
import pandas as pd

df = pd.read_parquet("/home/lixiangyu/xwq/Subcellular_Feature/data/xgb_7class_no_foci/multiclass_xgb_7class_no_foci_group5cv_oof.parquet")

same_count = (df["y_true"] == df["pattern_top1"]).sum()
total_count = len(df)

print("y_true 与 pattern_top1 相同的数目:", same_count)
print("总样本数:", total_count)
print("占比:", same_count / total_count)

y_true 与 pattern_top1 相同的数目: 16231
总样本数: 17500
占比: 0.9274857142857142


In [19]:
# 2) 不同的条目
diff_df = df[df["y_true"] != df["pattern_top1"]].copy()

print("y_true 与 pattern_top1 不同的条数:", len(diff_df))
diff_view = diff_df[["cell", "gene", "y_true", "pattern_top1"]].copy()
diff_view.head(50)

y_true 与 pattern_top1 不同的条数: 1269


,cell,gene,y_true,pattern_top1
1,cell_135,gene_0_0_1,Random,Cytoplasmic
5,cell_135,gene_0_1_0,Random,Nuclear
8,cell_135,gene_0_1_3,Random,Cytoplasmic
15,cell_135,gene_0_3_0,Random,Nuclear
22,cell_135,gene_0_4_2,Random,Cytoplasmic
32,cell_135,gene_0_6_2,Random,Cytoplasmic
35,cell_135,gene_0_7_0,Random,Cytoplasmic
50,cell_135,gene_2_0_0,Nuclear,Random
100,cell_135,gene_3_0_0,Cytoplasmic,Nuclear edge
102,cell_135,gene_3_0_2,Cytoplasmic,Random
